# CASMI 2026 training spectra: download and first look

Pilot for the overlap of the Enveda CASMI 2026 training structures with PlantMetWiki (plan in [../README.md](../README.md)). This notebook downloads the competition files and checks the columns. The files were downloaded after the competition rules were accepted.

## Citation

David Healey, Christoph Krettler, Tobias Kind, Marie Killian, Seth Drake, Erik DeBloois, Ivy Lightheart, Yojana Gadiya, Pelle Simpson, Daniel Domingo-Fernandez, Viswa Colluru, August Allen, Walter Reade, Ashley Oldacre. *Enveda CASMI 2026 - Molecule ID From Mass Spectra*. <https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra>, 2026. Kaggle.

Use of the data is under the competition rules; the raw files stay out of git (`data/` is git-ignored).

## Credentials

The Kaggle token is read from `../kaggle.env` (one line, `KAGGLE_API_TOKEN=...`). That file is git-ignored and must never be committed or printed.

In [1]:
import os
from pathlib import Path
import pandas as pd

HERE = Path.cwd().parent                       # pilots/casmi26_overlap (notebook runs from notebooks/)
for line in (HERE / "kaggle.env").read_text().splitlines():
    if "=" in line:
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()
os.environ["KAGGLEHUB_CACHE"] = str(HERE / "data")   # keep the files inside the (ignored) data folder
import kagglehub

/private/tmp/claude-501/-Users-elenadelpup-software-LOD-pathways/651ca791-07b8-4bf0-a246-1c66c202697b/scratchpad/kh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = Path(kagglehub.competition_download("enveda-CASMI26-molecule-id-mass-spectra"))
print("Path to competition files:", path)
sorted(p.name for p in path.iterdir())

Path to competition files: /Users/elenadelpup/software/LOD-pathways/lod-pathways-community/pilots/casmi26_overlap/data/competitions/enveda-CASMI26-molecule-id-mass-spectra


['sample_submission.csv', 'test.parquet', 'train.parquet']

## Columns

The description lists 18 columns for `train.parquet`, including `inchikey`, `inchikey14`, `normalized_smiles`, `molecular_formula`, `ingest_lib` and `precursor_error_ppm`. Check this against the file.

In [3]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(path / "train.parquet")
print(pf.metadata.num_rows, "rows;", pf.metadata.num_columns, "columns")
print(pf.schema_arrow.names)

2539608 rows; 18 columns
['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs', 'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity', 'collision_energy_ev', 'collision_energy_orig', 'collision_energy_orig_units']


## Unique structures per library

Read only the columns needed (the peak arrays are large).

In [4]:
tr = pd.read_parquet(path / "train.parquet", columns=["inchikey14", "ingest_lib", "precursor_error_ppm", "adduct"])
summary = tr.groupby("ingest_lib").agg(spectra=("inchikey14", "size"), structures=("inchikey14", "nunique")).sort_values("spectra", ascending=False)
summary.loc["all (unique inchikey14)"] = [len(tr), tr.inchikey14.nunique()]
summary

,spectra,structures
ingest_lib,,
enveda-180,1153785,182941
pluskal_ms2,527581,46821
riken,347171,15892
gnps,220849,45750
massbank,101727,9180
mona,92416,11681
spectraverse,50933,9631
msdial,40765,9127
drug_plus,2545,2539


## Overlap with PlantMetWiki (inchikey14)

Computed by [../casmi_overlap.py](../casmi_overlap.py) (reads only the key columns; results in `../results/`). The key is the first block of the InChIKey, so stereoisomers and charge forms of the same skeleton match.

In [5]:
import json
RES = HERE / "results"
s = json.loads((RES / "overlap_summary.json").read_text())
print(json.dumps(s, indent=1))
pd.read_csv(RES / "overlap_per_library.csv")

{
 "spectra": 2539608,
 "structures": 275810,
 "spectra_precursor_error_le_10ppm": 2360254,
 "structures_with_good_spectrum": 273530,
 "pmw_compounds": 2732,
 "pmw_compounds_in_pathway": 2384,
 "overlap_compounds": 1258,
 "overlap_compounds_in_pathway": 1145,
 "overlap_pct_of_pmw": 46.0,
 "n_libraries_per_compound": {
  "1": 172,
  "2": 124,
  "3": 132,
  "4": 139,
  "5": 194,
  "6": 248,
  "7": 192,
  "8": 47,
  "9": 10
 },
 "pathways": 1162,
 "pathways_with_any_spectrum": 1125,
 "pathways_ge_50pct_with_spectra_and_ge4_members": 782
}


,library,spectra,structures,pmw_matches,pmw_in_pathway,pct_of_structures,pmw_matches_good_spectrum
0,ALL,2539608,275810,1258,1145,0.46,1223
1,enveda-180,1153785,182941,0,0,0.00,0
2,pluskal_ms2,527581,46821,502,478,1.07,478
3,gnps,220849,45750,981,894,2.14,925
4,riken,347171,15892,1019,940,6.41,907
5,mona,92416,11681,829,760,7.10,772
6,spectraverse,50933,9631,20,15,0.21,20
7,massbank,101727,9180,882,806,9.61,753
8,msdial,40765,9127,848,781,9.29,847
9,drug_plus,2545,2539,442,400,17.41,424


### Reading

- **Structures with a spectrum:** 275,810 unique structures (`inchikey14`). 1,258 of 2,732 PlantMetWiki compounds (46 %) are among them; 1,145 of those are in a non-reaction pathway.
- **By library:** the synthetic drug-like `enveda-180` set (183k structures) matches **0** PlantMetWiki compounds. The matches come from the natural-product and metabolite libraries: RIKEN 1,019, GNPS 981, MassBank 882, MS-DIAL 848, MoNA 829, pluskal 502, `drug_plus` 442 (many matches are small primary metabolites that are also drugs). The smaller libraries have a higher percentage because they are small and focused.
- **Spectra are mostly from several libraries:** 1,086 of the 1,258 compounds have spectra in 2 or more libraries (see `n_libraries_per_compound`).
- **This is identity, not chance:** a matched key means the same skeleton. A shuffled null is not needed for the count; what matters is which pathways are covered.

In [6]:
cov = pd.read_csv(RES / "pathway_spectra_coverage.csv")
print(len(cov), "pathways;", (cov.with_spectra > 0).sum(), "have at least one compound with a reference spectrum;",
      ((cov.fraction >= 0.5) & (cov.members >= 4)).sum(), "have >= 4 members and >= half of them with spectra")
cov.head(15)

1162 pathways; 1125 have at least one compound with a reference spectrum; 782 have >= 4 members and >= half of them with spectra


,plantcyc_id,pathway,members,with_spectra,fraction
0,PWY-7345,superpathway of anaerobic sucrose degradation,67,45,0.672
1,PWY-5464,"superpathway of cytosolic glycolysis (plants),...",56,43,0.768
2,PWY-6266,superpathway of flavones and derivatives biosy...,64,37,0.578
3,PWY-1042,glycolysis IV,48,36,0.750
4,GLYCOLYSIS,glycolysis I (from glucose 6-phosphate),51,35,0.686
5,PWY-5484,glycolysis II (from fructose 6-phosphate),45,33,0.733
6,PWY-5723,Rubisco shunt,38,30,0.789
7,GLUCONEO-PWY,gluconeogenesis I,45,29,0.644
8,PWY-7341,superpathway of hyoscyamine (atropine) and sco...,43,28,0.651
9,P165-PWY,superpathway of purines degradation in plants,38,25,0.658


## The THT example: caffeoyltyramine has reference MS/MS

The hypothesis of the THT example (feature M300T887 = caffeoyltyramine, not in PlantMetWiki, no MS/MS in the MetaboLights files) can be checked against these libraries.

In [7]:
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchiKey
tr = pd.read_parquet(path / "train.parquet", columns=["inchikey14", "ingest_lib", "adduct", "collision_energy_ev", "precursor_mz"])
smi = {"caffeoyltyramine (hypothesis)": "O=C(/C=C/c1ccc(O)c(O)c1)NCCc1ccc(O)cc1", "p-coumaroyltyramine": "O=C(/C=C/c1ccc(O)cc1)NCCc1ccc(O)cc1",
       "feruloyltyramine": "O=C(/C=C/c1ccc(O)c(OC)c1)NCCc1ccc(O)cc1", "tyramine": "NCCc1ccc(O)cc1", "caffeic acid": "OC(=O)/C=C/c1ccc(O)c(O)c1"}
rows = []
for n, sm in smi.items():
    k = MolToInchiKey(Chem.MolFromSmiles(sm))[:14]; d = tr[tr.inchikey14 == k]
    rows.append({"compound": n, "inchikey14": k, "spectra": len(d), "libraries": ", ".join(sorted(set(d.ingest_lib))), "[M+H]+ spectra": int((d.adduct == "[M+H]+").sum())})
pd.DataFrame(rows)

,compound,inchikey14,spectra,libraries,[M+H]+ spectra
0,caffeoyltyramine (hypothesis),VSHUQLRHTJOKTA,22,"mona, pluskal_ms2, riken",11
1,p-coumaroyltyramine,RXGUTQNKCXHALN,100,"enveda-np-examples, gnps, massbank, mona, msdi...",54
2,feruloyltyramine,NPNNKDMSXVRADT,161,"gnps, massbank, mona, msdial, pluskal_ms2, riken",77
3,tyramine,DZGWFCGJZKJUFP,217,"drug_plus, gnps, massbank, mona, msdial, plusk...",198
4,caffeic acid,QAIPRVGONGVQAS,714,"drug_plus, enveda-np-examples, gnps, massbank,...",354


Caffeoyltyramine has 22 reference spectra (MoNA, pluskal_ms2, RIKEN), including [M+H]+. So the structure is **in the libraries although it is not in PlantMetWiki**. This makes the MS/MS check possible if the MS/MS of the feature were available (the public MetaboLights files have none). The reference spectra also give the fragments to look for: the tyramine and caffeoyl fragments, and the precursor at m/z 300.1230 for [M+H]+.

In [8]:
cov[cov.plantcyc_id == "PWY-5474"]

,plantcyc_id,pathway,members,with_spectra,fraction
407,PWY-5474,hydroxycinnamic acid tyramine amides biosynthesis,21,7,0.333
